# 🎯 YOLO Road Damage Detection Model Training & Taxonomy Pipeline
**Project:** NagarSeva-AI  
**Dataset Config:**   
**Target Output:**   
**Objective:** Train Ultralytics YOLO on RDD2022 dataset to detect road defects (, , , ), evaluate mAP metrics, save production model weights, and connect predictions to master taxonomy category ().

## 1. Environment & Hardware Diagnostic
Verify PyTorch version, check Apple Silicon MPS (Metal Performance Shaders) / GPU acceleration availability, and confirm Ultralytics installation.

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO
import yaml
import cv2
import matplotlib.pyplot as plt

print(f'PyTorch Version: {torch.__version__}')
print(f'Ultralytics Version: {ultralytics.__version__}')

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training Hardware Accelerator: {device.upper()}')

## 2. Dataset YAML Configuration Audit
Inspect  to confirm dataset root path, image split directories, and class label definitions.

In [ ]:
config_path = '../configs/dataset.yaml'
if not os.path.exists(config_path):
    config_path = 'configs/dataset.yaml'

with open(config_path, 'r') as f:
    dataset_cfg = yaml.safe_load(f)

print('Dataset Configuration:')
print(yaml.dump(dataset_cfg, default_flow_style=False))

## 3. Model Initialization & Pre-trained Weights
Initialize Ultralytics YOLO model ( / ).

In [ ]:
model = YOLO('yolo11n.pt')
print('YOLO Model Architecture Loaded Successfully.')

## 4. Train YOLO Model on RDD2022
Launch model training on RDD2022 dataset for 50 epochs at 640x640 resolution.

In [ ]:
results = model.train(
    data=config_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project='../runs/road_damage' if os.path.exists('../runs') else 'runs/road_damage',
    name='rdd2022_yolo'
)
print('Model Training Complete.')

## 5. Model Evaluation & Performance Metrics
Audit mAP@50, mAP@50-95, Precision, and Recall scores across all road damage classes.

In [ ]:
metrics = model.val()
print('=== YOLO Model Evaluation Metrics ===')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## 6. Save Production Model Weights
Save finalized  model weights to  for deployment.

In [ ]:
import shutil

dest_model_path = '../models/road_damage/best.pt'
if not os.path.exists('../models'):
    dest_model_path = 'models/road_damage/best.pt'

best_weights_src = os.path.join(model.trainer.save_dir, 'weights', 'best.pt')
if os.path.exists(best_weights_src):
    shutil.copy(best_weights_src, dest_model_path)
    print(f'Production model saved to: {dest_model_path}')

## 7. Two-Level Classification & FastAPI Payload Integration
Connect visual detection outputs (, , etc.) to NagarSeva-AI master taxonomy category () and format sample FastAPI JSON response.

In [ ]:
# Taxonomy Mapping Dictionary
yolo_to_nagarseva_taxonomy = {
    'pothole': 'ROAD_INFRASTRUCTURE',
    'longitudinal_crack': 'ROAD_INFRASTRUCTURE',
    'transverse_crack': 'ROAD_INFRASTRUCTURE',
    'alligator_crack': 'ROAD_INFRASTRUCTURE'
}

def predict_road_damage_api(image_path, model_instance):
    results = model_instance(image_path)
    detections = []
    
    for r in results:
        boxes = r.boxes
        for box in boxes:
            cls_id = int(box.cls[0])
            sub_cat = model_instance.names[cls_id]
            conf = float(box.conf[0])
            bbox = [round(x, 2) for x in box.xyxy[0].tolist()]
            
            # Determine severity heuristic based on bounding box area & class
            severity = 'High' if (sub_cat == 'pothole' or conf > 0.85) else 'Medium'
            
            detections.append({
                'detected': True,
                'category': yolo_to_nagarseva_taxonomy.get(sub_cat, 'ROAD_INFRASTRUCTURE'),
                'sub_category': sub_cat,
                'confidence': round(conf, 4),
                'severity': severity,
                'bounding_box': bbox
            })
    return detections

print('FastAPI Detection Pipeline Interface Ready.')

## 8. Summary & Key Takeaways

### Q&A
- **Where is the dataset config stored?** .
- **What accelerator is used?** Apple Silicon  (Metal Performance Shaders) / PyTorch GPU.
- **Where is the production model stored?** .

### Key Findings
- **Two-Level Taxonomy:** Level 1 predicts visual sub-category (, , etc.), while Level 2 maps seamlessly to  for unified application routing.
- **FastAPI Ready:** Output payload provides bounding box, confidence score, sub-category, severity, and master category for live civic issue creation.